# 04_sbert_official_experiment.ipynb
│  
├── 读取 raw data  
├── merge splits.csv  
├── 使用18 genres  
├── official train/val/test  
├── SBERT embeddings  
├── Logistic Regression  
├── threshold tuning  
├── test evaluation  
└── 保存统一预测文件  

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"

print(DATA_DIR)
print(RAW_DIR)

../data
../data/raw


In [2]:
train_raw = pd.read_csv(
    RAW_DIR / "train.csv"
)

splits = pd.read_csv(
    DATA_DIR / "splits.csv"
)

genre_mapping = pd.read_csv(
    DATA_DIR / "genre_mapping.csv"
)

print(train_raw.shape)
print(splits.shape)
print(genre_mapping.shape)

(8000, 4)
(7991, 2)
(18, 2)


In [3]:
df = train_raw.merge(
    splits,
    on="movie_id",
    how="inner"
)

print("Merged dataset shape:", df.shape)

display(df.head())

Merged dataset shape: (7991, 5)


,movie_id,title,overview,genre_ids,split
0,1162,They Call Me Trinity,The simple story has the pair coming to the re...,"[28, 35, 37]",validation
1,3425,Lone Star,When the skeleton of his murdered predecessor ...,"[18, 9648, 10749]",train
2,8515,AVP: Alien vs. Predator,When scientists discover something near Antarc...,"[12, 878, 28, 27]",train
3,3619,Wristcutters: A Love Story,"Zia, distraught over breaking up with his girl...","[35, 18, 14, 10749]",train
4,6080,The Pickup,A routine cash pickup takes a wild turn when m...,"[28, 35, 80]",train


In [4]:
print(df["split"].value_counts())

split
train         5594
validation    1199
test          1198
Name: count, dtype: int64


In [5]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (5594, 5)
Validation: (1199, 5)
Test: (1198, 5)


In [7]:
display(genre_mapping)
print(genre_mapping.columns)

,genre_id,genre_name
0,12,Adventure
1,14,Fantasy
2,16,Animation
3,18,Drama
4,27,Horror
5,28,Action
6,35,Comedy
7,36,History
8,37,Western
9,53,Thriller


Index(['genre_id', 'genre_name'], dtype='str')


In [8]:
import ast

def parse_genre_ids(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

train_df["genre_ids"] = train_df["genre_ids"].apply(parse_genre_ids)
val_df["genre_ids"] = val_df["genre_ids"].apply(parse_genre_ids)
test_df["genre_ids"] = test_df["genre_ids"].apply(parse_genre_ids)

print(train_df["genre_ids"].head())
print(type(train_df.iloc[0]["genre_ids"]))

1      [18, 9648, 10749]
2      [12, 878, 28, 27]
3    [35, 18, 14, 10749]
4           [28, 35, 80]
6     [10749, 18, 10402]
Name: genre_ids, dtype: object
<class 'list'>


In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

GENRE_IDS = genre_mapping["genre_id"].tolist()
GENRE_NAMES = genre_mapping["genre_name"].tolist()

mlb = MultiLabelBinarizer(
    classes=GENRE_IDS
)

In [10]:
print(len(GENRE_IDS))
print(GENRE_IDS)

18
[12, 14, 16, 18, 27, 28, 35, 36, 37, 53, 80, 878, 9648, 10402, 10749, 10751, 10752, 10770]


In [11]:
y_train = mlb.fit_transform(
    train_df["genre_ids"]
)

y_val = mlb.transform(
    val_df["genre_ids"]
)

y_test = mlb.transform(
    test_df["genre_ids"]
)

In [12]:
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

y_train: (5594, 18)
y_val: (1199, 18)
y_test: (1198, 18)


In [13]:
def vector_to_genres(vector):
    return [
        name
        for name, value in zip(
            GENRE_NAMES,
            vector
        )
        if value == 1
    ]


print(train_df.iloc[0]["title"])
print(train_df.iloc[0]["genre_ids"])

print(
    vector_to_genres(y_train[0])
)

Lone Star
[18, 9648, 10749]
['Drama', 'Mystery', 'Romance']


In [15]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    return " ".join(str(text).split())


train_texts = (
    train_df["overview"]
    .apply(normalize_text)
    .tolist()
)

val_texts = (
    val_df["overview"]
    .apply(normalize_text)
    .tolist()
)

test_texts = (
    test_df["overview"]
    .apply(normalize_text)
    .tolist()
)


print("Train texts:", len(train_texts))
print("Validation texts:", len(val_texts))
print("Test texts:", len(test_texts))

Train texts: 5594
Validation texts: 1199
Test texts: 1198


In [16]:
for i in range(3):
    print("="*50)
    print(train_df.iloc[i]["title"])
    print(train_texts[i][:300])

Lone Star
When the skeleton of his murdered predecessor is found, Sheriff Sam Deeds unearths many other long-buried secrets in his Texas border town.
AVP: Alien vs. Predator
When scientists discover something near Antarctica that appears to be a buried Pyramid, they send a research team out to investigate. Little do they know that they are about to step into a hunting ground where Aliens are grown as sport for the Predator race.
Wristcutters: A Love Story
Zia, distraught over breaking up with his girlfriend, decides to end it all. Unfortunately, he discovers that there is no real ending, only a run-down afterlife that is strikingly similar to his old one, just a bit worse. Discovering that his ex-girlfriend has also "offed" herself, he sets out on a 


In [17]:
import torch
from sentence_transformers import SentenceTransformer

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: mps


In [18]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print(
    "Embedding dimension:",
    sbert_model.get_sentence_embedding_dimension()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/var/folders/25/kk5ffyvs7c54btsfwts9g4lm0000gn/T/ipykernel_34524/3137118714.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sbert_model.get_sentence_embedding_dimension()


In [19]:
test_embedding = sbert_model.encode(
    train_texts[:5],
    batch_size=5,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(test_embedding.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(5, 384)


In [20]:
train_embeddings = sbert_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

val_embeddings = sbert_model.encode(
    val_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

test_embeddings = sbert_model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [21]:
from pathlib import Path

CACHE_DIR = Path("../data/processed")
CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [22]:
import numpy as np

np.save(
    CACHE_DIR / "sbert_train_embeddings.npy",
    train_embeddings
)

np.save(
    CACHE_DIR / "sbert_val_embeddings.npy",
    val_embeddings
)

np.save(
    CACHE_DIR / "sbert_test_embeddings.npy",
    test_embeddings
)

print("Saved.")

Saved.


In [23]:
print(train_embeddings.shape)
print(val_embeddings.shape)
print(test_embeddings.shape)

(5594, 384)
(1199, 384)
(1198, 384)


In [24]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

In [25]:
classifier = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        random_state=42
    )
)

In [29]:
classifier.fit(
    train_embeddings,
    y_train
)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](18,)","[ 0, 1, 2,...,15,16,17]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,18
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,384
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42


In [31]:
print("X:")
print(train_embeddings.shape)

print("Y:")
print(y_train.shape)

X:
(5594, 384)
Y:
(5594, 18)


In [32]:
val_scores = classifier.predict_proba(
    val_embeddings
)

print(val_scores.shape)

(1199, 18)


In [33]:
from sklearn.metrics import f1_score

val_pred_default = (
    val_scores >= 0.5
).astype(int)

macro_f1_default = f1_score(
    y_val,
    val_pred_default,
    average="macro",
    zero_division=0
)

micro_f1_default = f1_score(
    y_val,
    val_pred_default,
    average="micro",
    zero_division=0
)

print("Macro-F1:", macro_f1_default)
print("Micro-F1:", micro_f1_default)

Macro-F1: 0.44192142058010214
Micro-F1: 0.5634935836046734


In [34]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(
    0.1,
    0.81,
    0.05
)

results = []

for threshold in thresholds:
    preds = (
        val_scores >= threshold
    ).astype(int)

    macro = f1_score(
        y_val,
        preds,
        average="macro",
        zero_division=0
    )

    micro = f1_score(
        y_val,
        preds,
        average="micro",
        zero_division=0
    )

    results.append(
        {
            "threshold": threshold,
            "macro_f1": macro,
            "micro_f1": micro
        }
    )


threshold_results = pd.DataFrame(results)

display(threshold_results)

,threshold,macro_f1,micro_f1
0,0.10,0.493902,0.535903
1,0.15,0.534914,0.585623
2,0.20,0.542883,0.608591
3,0.25,0.551871,0.624917
4,0.30,0.539145,0.629453
5,0.35,0.530286,0.626394
6,0.40,0.505037,0.615104
7,0.45,0.479044,0.596930
8,0.50,0.441921,0.563494
9,0.55,0.406027,0.525393


In [35]:
best_row = threshold_results.loc[
    threshold_results["macro_f1"].idxmax()
]

best_threshold = best_row["threshold"]

print("Best threshold:", best_threshold)
print("Best Macro-F1:", best_row["macro_f1"])
print("Micro-F1:", best_row["micro_f1"])

Best threshold: 0.25000000000000006
Best Macro-F1: 0.5518713716771968
Micro-F1: 0.6249174263442991


## Validation Threshold Selection

Using the official split:

- Train: 5594
- Validation: 1199
- Test: 1198

The classifier outputs probabilities for 18 genres.

A threshold sweep from 0.10 to 0.80 was performed on the validation set.

Best threshold:
0.25

Validation performance:

- Macro-F1: 0.5519
- Micro-F1: 0.6249

In [36]:
test_scores = classifier.predict_proba(
    test_embeddings
)

test_pred = (
    test_scores >= best_threshold
).astype(int)

print(test_scores.shape)
print(test_pred.shape)

(1198, 18)
(1198, 18)


In [37]:
from sklearn.metrics import f1_score

test_macro_f1 = f1_score(
    y_test,
    test_pred,
    average="macro",
    zero_division=0
)

test_micro_f1 = f1_score(
    y_test,
    test_pred,
    average="micro",
    zero_division=0
)

print("Test Macro-F1:", test_macro_f1)
print("Test Micro-F1:", test_micro_f1)

Test Macro-F1: 0.5476366321890084
Test Micro-F1: 0.624383744170553


In [38]:
experiment_config = {
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dim": 384,
    "classifier": "OneVsRest Logistic Regression",
    "genres": 18,
    "train_size": 5594,
    "validation_size": 1199,
    "test_size": 1198,
    "threshold": float(best_threshold),
    "validation_macro_f1": float(best_row["macro_f1"]),
    "validation_micro_f1": float(best_row["micro_f1"]),
    "test_macro_f1": float(test_macro_f1),
    "test_micro_f1": float(test_micro_f1)
}

experiment_config

{'model': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_dim': 384,
 'classifier': 'OneVsRest Logistic Regression',
 'genres': 18,
 'train_size': 5594,
 'validation_size': 1199,
 'test_size': 1198,
 'threshold': 0.25000000000000006,
 'validation_macro_f1': 0.5518713716771968,
 'validation_micro_f1': 0.6249174263442991,
 'test_macro_f1': 0.5476366321890084,
 'test_micro_f1': 0.624383744170553}

In [39]:
import json

with open(
    "../data/processed/sbert_experiment_config.json",
    "w"
) as f:
    json.dump(
        experiment_config,
        f,
        indent=4
    )

In [40]:
import numpy as np
import pandas as pd

In [41]:
test_movie_ids = test_df["movie_id"].to_numpy()

print(test_movie_ids.shape)

(1198,)


In [43]:
n_movies = len(test_movie_ids)
n_genres = len(GENRE_IDS)

prediction_df = pd.DataFrame({

    "movie_id": np.repeat(
        test_movie_ids,
        n_genres
    ),

    "model": "sbert_lr",

    "variant": "original",

    "seed": 42,

    "genre_id": np.tile(
        GENRE_IDS,
        n_movies
    ),

    "y_true": y_test.reshape(-1),

    "y_score": test_scores.reshape(-1),

    "y_pred": test_pred.reshape(-1)
})

In [44]:
print(prediction_df.shape)

display(
    prediction_df.head(20)
)

(21564, 8)


,movie_id,model,variant,seed,genre_id,y_true,y_score,y_pred
0,1210,sbert_lr,original,42,12,0,0.070898,0
1,1210,sbert_lr,original,42,14,0,0.016000,0
2,1210,sbert_lr,original,42,16,0,0.022716,0
3,1210,sbert_lr,original,42,18,1,0.654789,1
4,1210,sbert_lr,original,42,27,0,0.050995,0
5,1210,sbert_lr,original,42,28,0,0.031304,0
6,1210,sbert_lr,original,42,35,0,0.605111,1
7,1210,sbert_lr,original,42,36,0,0.198905,0
8,1210,sbert_lr,original,42,37,0,0.004288,0
9,1210,sbert_lr,original,42,53,0,0.039158,0


In [45]:
from pathlib import Path

OUTPUT_DIR = Path("../predictions")
OUTPUT_DIR.mkdir(
    exist_ok=True
)

output_path = (
    OUTPUT_DIR /
    "sbert_original_predictions.csv"
)

prediction_df.to_csv(
    output_path,
    index=False
)

print(output_path)

../predictions/sbert_original_predictions.csv


In [46]:
print(
    prediction_df["model"].unique()
)

print(
    prediction_df["variant"].unique()
)

print(
    prediction_df["genre_id"].nunique()
)

<StringArray>
['sbert_lr']
Length: 1, dtype: str
<StringArray>
['original']
Length: 1, dtype: str
18


In [47]:
import sys

print(sys.executable)

/opt/anaconda3/envs/movie-sbert/bin/python
